# 3-4. Build Series & Exploratory Analysis

For each selected store: build a clean trading-day series (closed days dropped - see `src/timeseries.py`), then look at trend, weekly seasonality, promo/holiday effects, and stationarity. Corresponds to steps 3-4 of the workflow in `CLAUDE.md`.

In [ ]:
import sys, json, warnings
from pathlib import Path

SRC = Path.cwd().parent / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

import pandas as pd
import matplotlib.pyplot as plt
from statsmodels.tsa.stattools import adfuller, kpss

from data import load_merged
from timeseries import build_store_series
from plotting import set_paper_style, store_color, comma_axis, WEEKDAY_LABELS

set_paper_style()

OUTPUTS = Path.cwd().parent / "outputs"
FIGURES = OUTPUTS / "figures"
FIGURES.mkdir(parents=True, exist_ok=True)

with open(OUTPUTS / "selected_stores.json") as f:
    store_ids = json.load(f)

merged = load_merged()
series_by_store = {sid: build_store_series(merged, sid) for sid in store_ids}
store_ids

## Trend

In [ ]:
fig, axes = plt.subplots(len(store_ids), 1, figsize=(9, 2.2 * len(store_ids)), sharex=False)
for i, (sid, ax) in enumerate(zip(store_ids, axes)):
    s = series_by_store[sid]
    sales = s.frame["Sales"]
    rolling = sales.rolling(s.seasonal_period, center=True).mean()
    ax.plot(s.frame.index, sales, color=store_color(i), linewidth=1, alpha=0.3, label="Daily")
    ax.plot(s.frame.index, rolling, color=store_color(i), linewidth=2, label=f"{s.seasonal_period}-day rolling mean")
    ax.set_title(f"Store {sid}", loc="left", fontsize=9)
    ax.set_ylabel("Sales")
    comma_axis(ax, "y")
axes[0].legend(frameon=False, loc="upper right", fontsize=8)
axes[-1].set_xlabel("Trading day")
fig.tight_layout()
fig.savefig(FIGURES / "trend_by_store.png")
plt.show()

## Weekly seasonality

In [ ]:
fig, axes = plt.subplots(1, len(store_ids), figsize=(3.2 * len(store_ids), 3), sharey=False)
for i, (sid, ax) in enumerate(zip(store_ids, axes)):
    s = series_by_store[sid]
    days = sorted(s.frame["DayOfWeek"].unique())
    data = [s.frame.loc[s.frame["DayOfWeek"] == d, "Sales"] for d in days]
    ax.boxplot(
        data,
        tick_labels=[WEEKDAY_LABELS[d] for d in days],
        patch_artist=True,
        boxprops=dict(facecolor="#2a78d6", edgecolor="#0b0b0b"),
        medianprops=dict(color="#0b0b0b"),
    )
    ax.set_title(f"Store {sid}", fontsize=9)
    ax.set_xlabel("Day of week")
    comma_axis(ax, "y")
fig.tight_layout()
fig.savefig(FIGURES / "weekly_seasonality.png")
plt.show()

## Promo and holiday effects

In [ ]:
rows = []
for sid in store_ids:
    f = series_by_store[sid].frame
    rows.append({
        "store": sid,
        "mean_sales_promo": f.loc[f["Promo"] == 1, "Sales"].mean(),
        "mean_sales_no_promo": f.loc[f["Promo"] == 0, "Sales"].mean(),
        "mean_sales_school_holiday": f.loc[f["SchoolHoliday"] == 1, "Sales"].mean(),
        "mean_sales_no_school_holiday": f.loc[f["SchoolHoliday"] == 0, "Sales"].mean(),
    })
promo_effects = pd.DataFrame(rows).set_index("store")
promo_effects.to_csv(OUTPUTS / "tables" / "promo_holiday_effects.csv")
promo_effects

## Stationarity checks (ADF, KPSS)

In [ ]:
rows = []
with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    for sid in store_ids:
        y = series_by_store[sid].frame["Sales"]
        adf_stat, adf_p, *_ = adfuller(y)
        kpss_stat, kpss_p, *_ = kpss(y, nlags="auto")
        rows.append({
            "store": sid, "adf_stat": adf_stat, "adf_p": adf_p,
            "kpss_stat": kpss_stat, "kpss_p": kpss_p,
        })
stationarity = pd.DataFrame(rows).set_index("store")
stationarity.to_csv(OUTPUTS / "tables" / "stationarity.csv")
stationarity

ADF's null hypothesis is a unit root (non-stationary); KPSS's null is stationarity - using both guards against relying on a single test. Read the printed table above for the actual selected stores: a low ADF p-value (reject non-stationarity) together with a high KPSS p-value (fail to reject stationarity) is the consistent case for using SARIMA with a low differencing order; a disagreement between the two flags a store worth a second look (e.g. a trending or structurally-shifting series).